# Part 1 — Data Acquisition, Sampling, Cleaning, and Joins

**Owner:** Ziyue Xu (Daisy) · CIS 5450 Final Project · *Tipping Behavior in NYC Yellow Taxis*

This notebook builds the analysis table that every later part (EDA, hypothesis tests, modeling) reads.
It does five things:

1. Downloads the April–June 2026 NYC TLC yellow taxi trip records and the Taxi Zone Lookup Table.
2. Uses **DuckDB** to filter credit-card trips and draw a reproducible sample of 300,000 trips per month.
3. Cleans the sample with documented rules and records how many rows each rule removes.
4. **Joins** the trips to the zone lookup (pickup and drop-off) and to hourly Open-Meteo weather.
5. Builds the target (`high_tip`), the time-based split labels, and exports `trips_clean.parquet`.

> **Leakage rules enforced here.** `tip_amount`, `total_amount`, and `tip_pct` are kept only to build the target and must never be model features.
> No imputation, scaling, or encoding happens in this notebook; all of that is fitted on the training split inside the modeling pipeline.

## 0. Setup

In [ ]:
import os, sys, json, time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    !pip -q install duckdb

# Where raw and cleaned data live. On Colab this is a Google Drive folder, so files survive restarts.
DATA_DIR = Path(os.environ.get("CIS5450_DATA_DIR", "/content/drive/MyDrive/CIS5450_project/data"))
RAW_DIR, CLEAN_DIR = DATA_DIR / "raw", DATA_DIR / "clean"
RAW_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
print("DATA_DIR =", DATA_DIR)

In [ ]:
import duckdb
import numpy as np
import pandas as pd
import requests

pd.set_option("display.max_columns", 50)
print("duckdb", duckdb.__version__, "| pandas", pd.__version__)

### Parameters
All study design choices from the proposal live in one place. Change them here, never further down.

In [ ]:
SEED = 5450
MONTHS = ["2026-04", "2026-05", "2026-06"]
N_PER_MONTH = 300_000

STUDY_START, STUDY_END = pd.Timestamp("2026-04-01"), pd.Timestamp("2026-07-01")   # [start, end)
TRAIN_END = pd.Timestamp("2026-05-16")   # train: Apr 1 – May 15
VAL_END   = pd.Timestamp("2026-06-01")   # val:   May 16 – May 31; test: June

HIGH_TIP_THRESHOLD = 0.25          # main label, fixed before model fitting
SENSITIVITY_THRESHOLDS = [0.20, 0.30]
RAIN_MM = 0.1                      # rain indicator: precipitation >= 0.1 mm in the hour

NYC_LAT, NYC_LON = 40.7128, -74.0060
JFK, LGA, EWR = 132, 138, 1
UNKNOWN_ZONES = [264, 265]         # "Unknown" / "Outside of NYC"

# Optional: path to a sample you already drew earlier (CSV or Parquet).
# Leave as None to draw the sample again with DuckDB below.
EXISTING_SAMPLE_PATH = None

## 1. Data acquisition

Sources:
- [TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page): one Parquet file per month, one row per trip.
- Taxi Zone Lookup Table (CSV): `LocationID → Borough, Zone, service_zone`.

Files are downloaded once to Google Drive and reused afterwards, so the analysis does not depend on the TLC site staying unchanged.

In [ ]:
TLC_BASE = "https://d37ci6vzurychx.cloudfront.net"
FILES = {f"yellow_tripdata_{m}.parquet": f"{TLC_BASE}/trip-data/yellow_tripdata_{m}.parquet" for m in MONTHS}
FILES["taxi_zone_lookup.csv"] = f"{TLC_BASE}/misc/taxi_zone_lookup.csv"

def download(url, dest):
    if dest.exists() and dest.stat().st_size > 0:
        print(f"[cached] {dest.name}  ({dest.stat().st_size/1e6:.1f} MB)")
        return
    print(f"[download] {url}")
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        tmp = dest.with_suffix(dest.suffix + ".part")
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
        tmp.rename(dest)
    print(f"           -> {dest.stat().st_size/1e6:.1f} MB")

for name, url in FILES.items():
    download(url, RAW_DIR / name)

In [ ]:
con = duckdb.connect()

raw_summary = []
for m in MONTHS:
    path = str(RAW_DIR / f"yellow_tripdata_{m}.parquet")
    n_rows, n_card = con.execute(
        f"SELECT count(*), count(*) FILTER (WHERE payment_type = 1) FROM read_parquet('{path}')"
    ).fetchone()
    n_cols = len(con.execute(f"DESCRIBE SELECT * FROM read_parquet('{path}')").fetchall())
    raw_summary.append({"month": m, "rows": n_rows, "columns": n_cols,
                        "card_trips": n_card, "card_share": n_card / n_rows})
raw_summary = pd.DataFrame(raw_summary)
raw_summary

In [ ]:
# Column names and types of one raw file (all three months should match except request_source).
con.execute(f"DESCRIBE SELECT * FROM read_parquet('{RAW_DIR / f'yellow_tripdata_{MONTHS[-1]}.parquet'}')").df()[["column_name", "column_type"]]

## 2. Sampling with DuckDB

We keep only **credit-card trips** (`payment_type = 1`), because TLC records tips only for card payments; cash tips are always 0.
From each month we draw 300,000 card trips with reservoir sampling and a fixed seed, which gives 900,000 trips in total.

DuckDB queries the Parquet files directly, so the 11+ million raw rows are never loaded into pandas memory.

*Implementation note:* DuckDB applies `USING SAMPLE` before `WHERE`, so the card filter goes in a subquery and the sample is drawn from its result.
`SET threads = 1` makes the seeded sample exactly reproducible.

In [ ]:
def read_existing_sample(path):
    path = Path(path)
    return pd.read_parquet(path) if path.suffix == ".parquet" else pd.read_csv(path)

if EXISTING_SAMPLE_PATH:
    sample = read_existing_sample(EXISTING_SAMPLE_PATH)
    print("Loaded existing sample:", EXISTING_SAMPLE_PATH)
else:
    con.execute("SET threads = 1")
    parts = []
    for m in MONTHS:
        path = str(RAW_DIR / f"yellow_tripdata_{m}.parquet")
        part = con.execute(f'''
            SELECT * FROM (
                SELECT * FROM read_parquet('{path}', union_by_name = true)
                WHERE payment_type = 1
            ) USING SAMPLE reservoir({N_PER_MONTH} ROWS) REPEATABLE ({SEED})
        ''').df()
        part["source_month"] = m
        parts.append(part)
    con.execute("RESET threads")
    sample = pd.concat(parts, ignore_index=True)
    sample.to_parquet(RAW_DIR / "sample_card_trips.parquet", index=False)

for c in ["tpep_pickup_datetime", "tpep_dropoff_datetime"]:
    sample[c] = pd.to_datetime(sample[c])
if "source_month" not in sample:
    sample["source_month"] = sample["tpep_pickup_datetime"].dt.strftime("%Y-%m")

print(f"Sampled rows: {len(sample):,}")
print(sample["source_month"].value_counts().sort_index())
n_pos = ((sample["fare_amount"] > 0) & (sample["tip_amount"] > 0)).sum()
print(f"Positive fare and positive tip: {n_pos:,}   (proposal reports 825,072)")

### `request_source`
This column is not in the TLC data dictionary. We look at its values before deciding what to do with it.

In [ ]:
if "request_source" in sample:
    display(sample.groupby("source_month")["request_source"].value_counts(dropna=False).rename("n").to_frame())

As stated in the proposal, `request_source` is undocumented and empty (column absent) before June, so it cannot be used consistently across the train and test periods. We drop it.

In [ ]:
sample = sample.drop(columns=["request_source"], errors="ignore")

## 3. Cleaning

Each rule is applied in order and logged, so the notebook reports exactly how many rows each decision removes.

| # | Rule | Reason |
|---|---|---|
| 1 | `tip_amount > 0` | Study population. A zero card tip cannot distinguish "no tip" from "tipped in cash". |
| 2 | `fare_amount > 0` | Non-positive fares are refunds, voids, or errors, and the tip ratio is undefined. |
| 3 | Pickup between 2026-04-01 and 2026-06-30 | TLC files contain a few trips with timestamps outside their month. |
| 4 | 1 ≤ duration ≤ 180 minutes | Shorter trips are meter errors; longer ones are implausible for a single yellow cab trip. |
| 5 | 0 < `trip_distance` ≤ 100 miles | Zero distance is a meter error; > 100 miles is implausible within NYC service. |
| 6 | Pickup and drop-off zone not 264/265 | "Unknown" / "Outside of NYC" zones cannot be mapped to a borough. |
| 7 | `RatecodeID` ≠ 99 | 99 = unknown rate code. |
| 8 | `tip_pct` ≤ 100% | Tips above the fare are mostly data-entry errors and would dominate ratio summaries. |

Thresholds are fixed constants chosen from domain knowledge, not from quantiles of the data, so June (test) data does not influence any rule.
Missing values are **not** imputed here; that is done inside the modeling pipeline on training data only.

In [ ]:
df = sample.copy()
df["duration_min"] = (df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]).dt.total_seconds() / 60

RULES = [
    ("1. tip_amount > 0",                     lambda d: d["tip_amount"] > 0),
    ("2. fare_amount > 0",                    lambda d: d["fare_amount"] > 0),
    ("3. pickup within Apr 1 – Jun 30, 2026", lambda d: d["tpep_pickup_datetime"].between(STUDY_START, STUDY_END, inclusive="left")),
    ("4. 1 <= duration <= 180 min",           lambda d: d["duration_min"].between(1, 180)),
    ("5. 0 < trip_distance <= 100 mi",        lambda d: (d["trip_distance"] > 0) & (d["trip_distance"] <= 100)),
    ("6. PU/DO zone not 264/265",             lambda d: ~d["PULocationID"].isin(UNKNOWN_ZONES) & ~d["DOLocationID"].isin(UNKNOWN_ZONES)),
    ("7. RatecodeID != 99",                   lambda d: d["RatecodeID"].ne(99) | d["RatecodeID"].isna()),
    ("8. tip_pct <= 100%",                    lambda d: d["tip_amount"] / d["fare_amount"] <= 1.0),
]

funnel = [{"step": "0. sampled card trips", "rows_after": len(df), "removed": 0}]
for name, rule in RULES:
    before = len(df)
    df = df[rule(df).fillna(False).astype(bool)]
    funnel.append({"step": name, "rows_after": len(df), "removed": before - len(df)})
funnel = pd.DataFrame(funnel)
funnel["pct_removed_of_sample"] = (funnel["removed"] / funnel.loc[0, "rows_after"] * 100).round(2)
df = df.reset_index(drop=True)
funnel

In [ ]:
# Passenger count: 0 is not a valid count. Mark it missing instead of dropping the trip;
# the modeling pipeline imputes or encodes it as "unknown" using training data only.
n_zero_pass = int((df["passenger_count"] == 0).sum())
df.loc[df["passenger_count"] == 0, "passenger_count"] = np.nan
print(f"passenger_count == 0 set to missing: {n_zero_pass:,}")

missing = df.isna().sum()
missing[missing > 0].rename("missing").to_frame().assign(pct=lambda t: (t["missing"] / len(df) * 100).round(2))

In [ ]:
assert len(df) >= 50_000, "Course requirement: at least 50,000 rows after cleaning"
print(f"Rows after cleaning: {len(df):,}")

## 4. Join 1 — Taxi zones (pickup and drop-off)

`PULocationID` and `DOLocationID` are joined to the zone lookup twice, once per end of the trip.
We use LEFT JOINs in DuckDB and then check that the row count did not change and that every trip found a borough.

In [ ]:
zones = pd.read_csv(RAW_DIR / "taxi_zone_lookup.csv")
display(zones.head())

n_before = len(df)
df = con.execute('''
    SELECT t.*,
           pz.Borough AS PU_Borough, pz.Zone AS PU_Zone, pz.service_zone AS PU_service_zone,
           dz.Borough AS DO_Borough, dz.Zone AS DO_Zone, dz.service_zone AS DO_service_zone
    FROM df AS t
    LEFT JOIN zones AS pz ON t.PULocationID = pz.LocationID
    LEFT JOIN zones AS dz ON t.DOLocationID = dz.LocationID
''').df()

assert len(df) == n_before, "zone join changed the row count"
print(f"Rows before/after zone join: {n_before:,} / {len(df):,}")
print("Missing PU_Borough:", df["PU_Borough"].isna().sum(), "| Missing DO_Borough:", df["DO_Borough"].isna().sum())
df["PU_Borough"].value_counts()

In [ ]:
# Airport zones used later (H3 and the airport feature). Newark is kept in the data but excluded from H3.
pd.Series({
    "pickup or drop-off JFK":       df["PULocationID"].eq(JFK).sum() + df["DOLocationID"].eq(JFK).sum(),
    "pickup or drop-off LaGuardia": df["PULocationID"].eq(LGA).sum() + df["DOLocationID"].eq(LGA).sum(),
    "pickup or drop-off Newark":    (df["PULocationID"].eq(EWR) | df["DOLocationID"].eq(EWR)).sum(),
}, name="trip-ends")

## 5. Join 2 — Hourly weather (Open-Meteo)

Source: [Open-Meteo Historical Weather API](https://open-meteo.com/en/docs/historical-weather-api), hourly `temperature_2m` (°C) and `precipitation` (mm) at Manhattan coordinates, in New York local time.

**Alignment rule (no future information).** Open-Meteo labels hour *T* with precipitation accumulated from *T−1h* to *T*.
We floor each pickup time to the hour (`pickup_hour_ts`) and join on it, which selects the most recent hour that ended no later than pickup.
Rounding to the nearest hour instead would sometimes use an hour that had not finished yet.

TLC timestamps are New York local time without a time zone, which matches `timezone=America/New_York`. April–June 2026 has no daylight-saving transition (it happened on March 8), so every local hour is unique.

In [ ]:
WEATHER_PATH = RAW_DIR / "weather_hourly_nyc.csv"
if not WEATHER_PATH.exists():
    resp = requests.get("https://archive-api.open-meteo.com/v1/archive", timeout=60, params={
        "latitude": NYC_LAT, "longitude": NYC_LON,
        "start_date": "2026-04-01", "end_date": "2026-06-30",
        "hourly": "temperature_2m,precipitation", "timezone": "America/New_York",
    })
    resp.raise_for_status()
    pd.DataFrame(resp.json()["hourly"]).to_csv(WEATHER_PATH, index=False)

weather = pd.read_csv(WEATHER_PATH, parse_dates=["time"]).rename(
    columns={"time": "weather_hour", "temperature_2m": "temp_c", "precipitation": "precip_mm"})

expected_hours = (STUDY_END - STUDY_START) // pd.Timedelta(hours=1)
print(f"Weather hours: {len(weather):,} (expected {expected_hours:,}) | duplicates: {weather['weather_hour'].duplicated().sum()}")
print(weather[["temp_c", "precip_mm"]].isna().sum().to_dict())
assert len(weather) == expected_hours and not weather["weather_hour"].duplicated().any()
weather.describe()

In [ ]:
df["pickup_hour_ts"] = df["tpep_pickup_datetime"].dt.floor("h")

n_before = len(df)
df = con.execute('''
    SELECT t.*, w.temp_c, w.precip_mm
    FROM df AS t
    LEFT JOIN weather AS w ON t.pickup_hour_ts = w.weather_hour
''').df()
assert len(df) == n_before, "weather join changed the row count"

df["is_rain"] = (df["precip_mm"] >= RAIN_MM).astype("Int8").where(df["precip_mm"].notna())
coverage = df["temp_c"].notna().mean()
print(f"Weather join coverage: {coverage:.2%}")
print(f"Trips in rainy hours (>= {RAIN_MM} mm): {df['is_rain'].sum():,} ({df['is_rain'].mean():.2%})")
print(f"Rainy hours in the period: {(weather['precip_mm'] >= RAIN_MM).sum()} of {len(weather)}")

## 6. Derived features (proposal Table 1)

Every feature below is computed from the trip's own record, its zones, or the weather hour that ended before pickup, so it is known at trip completion and uses no information from other trips.
No statistic is learned from the data here, so building these before the split cannot leak test information.

| Column | Table 1 no. | Definition |
|---|---|---|
| `pickup_hour` | 1 | Hour of pickup, 0–23 |
| `pickup_dow` | 2 | Day of week, 0 = Monday … 6 = Sunday |
| `duration_min` | 3 | Built in Section 3 |
| `is_weekend` | 14 | Pickup on Saturday or Sunday |
| `is_late_night` | 15 | Pickup 00:00–05:59 (H1 group) |
| `avg_speed_mph` | 16 | 60 × `trip_distance` / `duration_min` |
| `is_airport` | 17 | Pickup or drop-off at JFK (132) or LaGuardia (138) (H3 group) |
| `same_borough` | 18 | `PU_Borough == DO_Borough` |
| `has_cbd_fee` | 19 | `cbd_congestion_fee > 0` |
| `temp_c`, `is_rain` | 20–21 | Built in Section 5 |

Two helper columns support the hypothesis tests: `is_newark` (Newark trips are excluded from H3) and `is_short_trip` (0 < distance ≤ 2 miles, H4).

In [ ]:
pu = df["tpep_pickup_datetime"]
df["pickup_hour"]   = pu.dt.hour.astype("int8")
df["pickup_dow"]    = pu.dt.dayofweek.astype("int8")
df["is_weekend"]    = (df["pickup_dow"] >= 5).astype("int8")
df["is_late_night"] = (df["pickup_hour"] <= 5).astype("int8")
df["avg_speed_mph"] = 60 * df["trip_distance"] / df["duration_min"]
df["is_airport"]    = (df["PULocationID"].isin([JFK, LGA]) | df["DOLocationID"].isin([JFK, LGA])).astype("int8")
df["is_newark"]     = (df["PULocationID"].eq(EWR) | df["DOLocationID"].eq(EWR)).astype("int8")
df["same_borough"]  = (df["PU_Borough"] == df["DO_Borough"]).astype("int8")
df["has_cbd_fee"]   = (df["cbd_congestion_fee"].fillna(0) > 0).astype("int8")
df["is_short_trip"] = (df["trip_distance"] <= 2).astype("int8")

flags = ["is_weekend", "is_late_night", "is_airport", "is_newark", "same_borough", "has_cbd_fee", "is_short_trip"]
display(df[flags].mean().rename("share of trips").to_frame().style.format("{:.1%}"))
df["avg_speed_mph"].describe()

In [ ]:
# Speeds above ~80 mph point to GPS or meter errors. We report them here; whether to cap or drop
# them is decided in feature engineering using the training split only.
n_fast = int((df["avg_speed_mph"] > 80).sum())
print(f"Trips with average speed > 80 mph: {n_fast:,} ({n_fast / len(df):.3%})")

## 7. Target, split labels, and export

- `tip_pct = tip_amount / fare_amount` (denominator excludes taxes, tolls, and surcharges).
- `high_tip = 1` if `tip_pct ≥ 25%`; `high_tip_20` and `high_tip_30` are kept for sensitivity checks only.
- `split` follows the proposal: train = Apr 1–May 15, val = May 16–31, test = June.
- `pickup_date` is the cluster unit for the day-clustered tests and day-block bootstrap.

In [ ]:
df["tip_pct"] = df["tip_amount"] / df["fare_amount"]
df["high_tip"] = (df["tip_pct"] >= HIGH_TIP_THRESHOLD).astype("int8")
for t in SENSITIVITY_THRESHOLDS:
    df[f"high_tip_{int(t * 100)}"] = (df["tip_pct"] >= t).astype("int8")

df["pickup_date"] = df["tpep_pickup_datetime"].dt.normalize()
df["split"] = np.select(
    [df["tpep_pickup_datetime"] < TRAIN_END, df["tpep_pickup_datetime"] < VAL_END],
    ["train", "val"], default="test")

df = df.sort_values("tpep_pickup_datetime").reset_index(drop=True)

split_summary = (df.groupby("split")
                   .agg(rows=("high_tip", "size"), days=("pickup_date", "nunique"),
                        high_tip_rate=("high_tip", "mean"),
                        first=("pickup_date", "min"), last=("pickup_date", "max"))
                   .reindex(["train", "val", "test"]))
split_summary

### Columns that must not be used as model features
Kept only to build the target or for bookkeeping. The modeling notebook should drop them explicitly.

In [ ]:
NOT_FEATURES = ["tip_amount", "total_amount", "tip_pct", "high_tip", "high_tip_20", "high_tip_30",
                "payment_type", "tpep_dropoff_datetime", "split", "source_month"]
print([c for c in NOT_FEATURES if c in df.columns])

In [ ]:
DATA_VERSION = "v1"
out_path = CLEAN_DIR / f"trips_clean_{DATA_VERSION}.parquet"
df.to_parquet(out_path, index=False)
funnel.to_csv(CLEAN_DIR / f"cleaning_funnel_{DATA_VERSION}.csv", index=False)
print(f"Saved {len(df):,} rows x {df.shape[1]} columns -> {out_path}  ({out_path.stat().st_size/1e6:.1f} MB)")

# CSV copies for viewing in Excel / Google Sheets. The Parquet file stays the source for analysis
# because it keeps column types (datetimes, nullable integers); CSV turns them into text.
csv_path = CLEAN_DIR / f"trips_clean_{DATA_VERSION}.csv"
df.to_csv(csv_path, index=False)
print(f"Saved full CSV -> {csv_path}  ({csv_path.stat().st_size/1e6:.1f} MB)")

preview_path = CLEAN_DIR / f"trips_clean_{DATA_VERSION}_preview_10k.csv"
df.sample(n=min(10_000, len(df)), random_state=SEED).sort_values("tpep_pickup_datetime").to_csv(preview_path, index=False)
print(f"Saved 10,000-row random preview CSV -> {preview_path}  (opens quickly in Excel or Sheets)")
df.dtypes.rename("dtype").to_frame()

## Summary

- Raw data: three monthly TLC Parquet files (see the table in Section 1); credit-card trips sampled with DuckDB at 300,000 per month.
- Cleaning: eight documented rules; the funnel table in Section 3 shows rows removed at each step.
- Joins: pickup and drop-off zones (row count unchanged, borough coverage checked) and hourly weather aligned to the last completed hour before pickup (coverage reported in Section 5).
- Features: the Table 1 derived features (Section 6) are already in the table.
- Output: `trips_clean_v1.parquet` (for analysis), plus `trips_clean_v1.csv` and a 10,000-row `trips_clean_v1_preview_10k.csv` for viewing, with the target `high_tip`, sensitivity labels, `split`, and `pickup_date`. Later notebooks read this file and never re-clean the data.

*Fill in the final numbers after the first full run.*